# 🎬 Movie Scene Voice Swap — Colab

Upload a movie clip + your own 30-second voice recording, and get the scene back
with the dialogue in **your** voice.

**How to run**
1. `Runtime → Change runtime type → T4 GPU`, then `Save`.
2. `Runtime → Run all` (or run Cell 1, then Cell 2, in order).
3. Wait for Cell 1 to finish installing (a few minutes).
4. Cell 2 prints a `something.gradio.live` link — open it, upload a clip, give your voice
   (record or upload an audio file), and click **Convert**.

This notebook already includes fixes for the version conflicts that commonly break
this pipeline on Colab (protobuf, TensorFlow auto-loading, a Seed-VC/huggingface_hub
compatibility bug). If Cell 1 or Cell 2 still errors, copy the **full** error text
back to get it fixed.

In [ ]:
# ============================================================
# CELL 1 — Install everything (run once per fresh runtime)
# ============================================================
!nvidia-smi | head -n 12

!git clone -q https://github.com/Plachtaa/seed-vc /content/seed-vc
%cd /content/seed-vc

# Colab already ships a working torch/numpy/gradio for this GPU+CUDA combo.
# Seed-VC's requirements.txt pins much older versions of those that don't
# even exist for this Python build, which breaks pip's resolver. So we drop
# ALL version pins and let pip fit everything to what's already installed.
import re
cleaned = []
for line in open("requirements.txt", encoding="utf-8"):
    name = line.strip().lower()
    if not name or name.startswith("#"):
        continue
    if name.startswith(("torch", "numpy", "gradio")):
        continue  # keep Colab's own versions of these
    cleaned.append(re.sub(r"(==|>=|<=|~=).*", "", line).strip())
open("/content/req_clean.txt", "w", encoding="utf-8").write("\n".join(cleaned))

!pip install -q -r /content/req_clean.txt
!pip install -q demucs torchcodec
# Seed-VC pulls in an old protobuf via one of its deps; Colab's own TensorFlow
# and related packages need a modern one. Put the modern one back on top.
!pip install -q -U "protobuf>=5.29,<7"

print("\n=== Install finished ===")

In [ ]:
# ============================================================
# CELL 2 — Launch the app (rerun this any time; no reinstall needed)
# ============================================================
import os, glob, re, subprocess, uuid, sys
import gradio as gr

# Keep Seed-VC / transformers from ever importing TensorFlow. Colab's TF build
# needs a newer protobuf than some of Seed-VC's own deps want, and importing
# TF at all is what triggers that clash — so just don't import it.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

SEEDVC = "/content/seed-vc"
WORK = "/content/work"
PY = sys.executable
os.makedirs(WORK, exist_ok=True)


def patch_seedvc():
    """Fix a real Seed-VC bug: its BigVGAN loader still requires the
    'proxies' and 'resume_download' keyword arguments that newer versions of
    huggingface_hub stopped passing in. This makes both arguments optional
    wherever Seed-VC defines a matching _from_pretrained(...) method."""
    patched_any = False
    for root, _, files in os.walk(SEEDVC):
        for fname in files:
            if not fname.endswith(".py"):
                continue
            path = os.path.join(root, fname)
            try:
                text = open(path, encoding="utf-8").read()
            except Exception:
                continue
            if "def _from_pretrained(" not in text:
                continue
            start = text.index("def _from_pretrained(")
            end_match = re.compile(r"\)\s*(?:->[^:]+)?:").search(text, start)
            if not end_match:
                continue
            sig = text[start:end_match.start()]
            new_sig = re.sub(
                r"\b(proxies|resume_download)(\s*:[^,=]*?)?\s*,",
                lambda m: f"{m.group(1)}{m.group(2) or ''} = None,",
                sig,
            )
            if new_sig != sig:
                open(path, "w", encoding="utf-8").write(text[:start] + new_sig + text[end_match.start():])
                patched_any = True
                print("Patched:", path)
    if not patched_any:
        print("No patch needed (already patched or file layout changed).")

patch_seedvc()


def run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print("=== FULL ERROR LOG ===")
        print(r.stdout[-3000:])
        print(r.stderr)
        tail = [l for l in r.stderr.strip().splitlines() if l.strip()][-6:]
        raise gr.Error("Step failed (" + " ".join(cmd[:2]) + "):  " + " | ".join(tail))
    return r


def convert(video, voice, steps, cfg_rate, progress=gr.Progress()):
    print("DEBUG video:", repr(video))
    print("DEBUG voice:", repr(voice))
    if not video:
        raise gr.Error("Please upload a video/clip.")
    if not voice:
        raise gr.Error("The app did not receive an audio file. Try uploading a recording "
                        "(mp3/wav/m4a) instead of using the mic button, or record again and "
                        "wait a couple of seconds after pressing stop before clicking Convert.")

    d = os.path.join(WORK, uuid.uuid4().hex[:8])
    os.makedirs(d)

    progress(0.05, desc="Extracting audio from video")
    orig = f"{d}/orig.wav"
    run(["ffmpeg", "-y", "-i", video, "-vn", "-ac", "2", "-ar", "44100", orig])

    ref = f"{d}/ref.wav"
    run(["ffmpeg", "-y", "-i", voice, "-t", "30", "-ac", "1", "-ar", "22050", ref])

    progress(0.15, desc="Separating dialogue from music/effects (Demucs)")
    run([PY, "-m", "demucs", "--two-stems=vocals", "-n", "htdemucs", "-o", f"{d}/sep", orig])
    sepdir = f"{d}/sep/htdemucs/orig"
    vocals, background = f"{sepdir}/vocals.wav", f"{sepdir}/no_vocals.wav"
    vocals_mono = f"{d}/vocals_mono.wav"
    run(["ffmpeg", "-y", "-i", vocals, "-ac", "1", "-ar", "22050", vocals_mono])

    progress(0.4, desc="Converting dialogue to your voice (Seed-VC) — first run downloads models")
    outdir = f"{d}/vc"
    os.makedirs(outdir)
    run([PY, "inference.py", "--source", vocals_mono, "--target", ref,
         "--output", outdir, "--diffusion-steps", str(int(steps)),
         "--length-adjust", "1.0", "--inference-cfg-rate", str(cfg_rate), "--fp16", "True"],
        cwd=SEEDVC)
    outs = sorted(glob.glob(f"{outdir}/*.wav"), key=os.path.getmtime)
    if not outs:
        raise gr.Error("Seed-VC produced no output file.")
    converted = outs[-1]

    progress(0.9, desc="Mixing your voice back into the scene")
    final = f"{d}/final.mp4"
    filter_complex = (
        "[1:a]aresample=44100,aformat=channel_layouts=stereo[v];"
        "[2:a]aresample=44100,aformat=channel_layouts=stereo[b];"
        "[v][b]amix=inputs=2:normalize=0:duration=longest[a]"
    )
    run(["ffmpeg", "-y", "-i", video, "-i", converted, "-i", background,
         "-filter_complex", filter_complex, "-map", "0:v:0", "-map", "[a]",
         "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", final])

    return final, converted


with gr.Blocks(title="Voice Swap") as demo:
    gr.Markdown(
        "# 🎬 Movie Scene Voice Swap\n"
        "Upload a clip, give 20-30 s of your voice (record or upload), get the scene back in your voice.\n\n"
        "**For the closest match:** record somewhere quiet, speak continuously for the full 30 s "
        "(avoid long pauses/short sentences), keep the mic steady and close, and use a natural pace."
    )
    with gr.Row():
        with gr.Column():
            video_in = gr.Video(label="1. Movie clip")
            voice_in = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="2. Your voice — record 20-30 s, or upload an mp3/wav/m4a (only the first 30 s is used)",
            )
            steps_in = gr.Slider(10, 100, value=90, step=5, label="Quality (more steps = closer match, slower)")
            cfg_in = gr.Slider(0.3, 1.0, value=0.95, step=0.05, label="Voice strength (higher = closer to your voice; too high can sound less smooth)")
            convert_btn = gr.Button("Convert", variant="primary")
        with gr.Column():
            video_out = gr.Video(label="Result (use the download icon)")
            audio_out = gr.Audio(label="Converted dialogue only")
    convert_btn.click(convert, [video_in, voice_in, steps_in, cfg_in], [video_out, audio_out])

demo.queue().launch(share=True, debug=True, allowed_paths=[WORK])